<a href="https://colab.research.google.com/github/Godstouch/GNN-Student-Risk-Prediction-/blob/main/Real_graph_corrected.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
!{sys.executable} -m pip install torch_geometric
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch_geometric.nn import GATConv
from sklearn.metrics import f1_score, classification_report, recall_score

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = torch.load('/content/real_graph_corrected.pt', weights_only=False).to(device)
num_classes = int(data.y.max().item()) + 1
num_features = data.x.shape[1]

class GATBaseline(nn.Module):

    def __init__(self, in_dim, hidden_dim, out_dim, heads=8, dropout=0.5):
        super().__init__()
        self.gat1 = GATConv(in_dim, hidden_dim, heads=heads, dropout=dropout)
        self.gat2 = GATConv(hidden_dim * heads, hidden_dim, heads=1, concat=False, dropout=dropout)
        self.skip = nn.Linear(in_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        h = F.dropout(x, p=self.dropout, training=self.training)
        h = F.elu(self.gat1(h, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.gat2(h, edge_index)
        s = F.relu(self.skip(x))
        z = F.dropout(h + s, p=self.dropout, training=self.training)
        return self.out(z)

def train_gnn(model, data, epochs=300, lr=0.01, weight_decay=5e-4, patience=40, verbose=True, class_weights=None):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val_f1 = -1; best_state = None; patience_ctr = 0
    history = {'train_loss': [], 'val_f1': []}
    for epoch in range(1, epochs + 1):
        model.train(); optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], weight=class_weights)
        loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            pred = out[data.val_mask].argmax(dim=1).cpu().numpy()
            true = data.y[data.val_mask].cpu().numpy()
            val_f1 = f1_score(true, pred, average='macro')
        history['train_loss'].append(loss.item()); history['val_f1'].append(val_f1)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
        if verbose and epoch % 20 == 0:
            print(f"Epoch {epoch}: loss={loss.item():.4f}, val_macro_f1={val_f1:.4f}")
        if patience_ctr >= patience:
            if verbose: print(f"Early stopping at epoch {epoch}, best val macro-F1={best_val_f1:.4f}")
            break
    model.load_state_dict(best_state)
    return history

def evaluate(model, data, mask):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out[mask].argmax(dim=1).cpu().numpy()
        true = data.y[mask].cpu().numpy()
    macro_f1 = f1_score(true, pred, average='macro')
    per_class = f1_score(true, pred, average=None)
    per_class_f1 = dict(zip([f'class_{i}' for i in range(num_classes)], per_class))
    high_risk_recall = recall_score(true, pred, labels=[0], average='macro')
    return {'macro_f1': macro_f1, 'per_class_f1': per_class_f1, 'high_risk_recall': high_risk_recall,
            'report': classification_report(true, pred, digits=3)}

set_seed(42)
counts = torch.bincount(data.y[data.train_mask])
class_weights = (counts.sum() / (num_classes * counts.float())).to(device)

model_gat = GATBaseline(num_features, hidden_dim=32, out_dim=num_classes, heads=8, dropout=0.5).to(device)
print("Training GATBaseline model...")
history_gat = train_gnn(model_gat, data, epochs=300, lr=0.01, weight_decay=5e-4, patience=40, verbose=True, class_weights=class_weights)
print("Training complete.")
eval_results_gat = evaluate(model_gat, data, data.test_mask)
print(f"Macro-F1 (GATBaseline): {eval_results_gat['macro_f1']:.4f}")
for cn, f1 in eval_results_gat['per_class_f1'].items():
    print(f"  {cn}: {f1:.4f}")
print(f"High-Risk Recall: {eval_results_gat['high_risk_recall']:.4f}")
print(eval_results_gat['report'])

Training GATBaseline model...
Epoch 20: loss=0.9826, val_macro_f1=0.3309
Epoch 40: loss=0.8785, val_macro_f1=0.3035
Early stopping at epoch 41, best val macro-F1=0.3672
Training complete.
Macro-F1 (GATBaseline): 0.2425
  class_0: 0.0417
  class_1: 0.4359
  class_2: 0.2500
High-Risk Recall: 0.0263
              precision    recall  f1-score   support

           0      0.100     0.026     0.042        38
           1      0.354     0.567     0.436        60
           2      0.273     0.231     0.250        52

    accuracy                          0.313       150
   macro avg      0.242     0.275     0.243       150
weighted avg      0.262     0.313     0.272       150

